# Red Wine Quality Prediction**Dataset**: UCI Wine Quality — Red Wine (1,599 samples, 11 features)  **Goal**: Classify wines as High or Low quality using Decision Tree, Logistic Regression, and Random Forest  **Best Model**: Random Forest — 82.2% accuracy, 0.91 AUC-ROCThis notebook covers: EDA → Preprocessing → Model Training & Tuning → Evaluation → Explainability (Feature Importance, ICE, PDP).

## 1. Setup & Data Loading

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport randomimport matplotlib.lines as mlinesfrom sklearn.preprocessing import MinMaxScalerfrom sklearn.model_selection import train_test_split, GridSearchCVfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, aucfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.linear_model import LogisticRegression%matplotlib inlinesns.set_style("whitegrid")plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load dataset (semicolon-delimited)df = pd.read_csv('data/winequality-red.csv', delimiter=';')print(f"Dataset shape: {df.shape}")df.head()

## 2. Exploratory Data Analysis

### 2.1 Data Overview

In [ ]:
df.info()print(f"\n{'='*50}")print(df.describe())

In [ ]:
# Check for missing valuesmissing = df.isnull().sum()print("Missing values per column:")print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

In [ ]:
# Check for outliers (example: residual sugar)Q1 = df['residual sugar'].quantile(0.25)Q3 = df['residual sugar'].quantile(0.75)IQR = Q3 - Q1outliers = df[(df['residual sugar'] < Q1 - 1.5 * IQR) | (df['residual sugar'] > Q3 + 1.5 * IQR)]print(f"Outliers in 'residual sugar': {len(outliers)}")

In [ ]:
# Quality score distributionprint("Quality Score Distribution:")print(df['quality'].value_counts().sort_index())

### 2.2 Visualizations

In [ ]:
# Box plots for all featuresfig, axes = plt.subplots(4, 3, figsize=(18, 14))for i, col in enumerate(df.drop('quality', axis=1).columns):    ax = axes[i // 3, i % 3]    sns.boxplot(x=df[col], ax=ax, color='steelblue')    ax.set_title(col, fontsize=12, fontweight='bold')plt.suptitle('Feature Distributions (Box Plots)', fontsize=16, fontweight='bold', y=1.01)plt.tight_layout()plt.show()

In [ ]:
# Correlation heatmapplt.figure(figsize=(12, 8))sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm', center=0, linewidths=0.5)plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# Feature correlations with quality (sorted)print("Feature Correlations with Quality:")print(df.corr()['quality'].sort_values())

In [ ]:
# Violin plot: quality vs alcoholplt.figure(figsize=(10, 6))sns.violinplot(x='quality', y='alcohol', data=df, palette='muted')plt.title('Wine Quality by Alcohol Content', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# Histograms for all featuresfig = df.hist(bins=15, figsize=(15, 12), edgecolor='white', color='steelblue')plt.suptitle('Feature Distributions (Histograms)', fontsize=16, fontweight='bold', y=1.01)plt.tight_layout()plt.show()

## 3. Data Preprocessing

In [ ]:
# Normalize features using MinMaxScalerscaler = MinMaxScaler()X = scaler.fit_transform(df.drop('quality', axis=1))# Binary classification: quality > 5 → "High", else → "Low"df['rating'] = np.where(df['quality'] > 5, 'High', 'Low')y = df['rating']print(f"Class distribution:")print(y.value_counts())print(f"\nFeature matrix shape: {X.shape}")

In [ ]:
# Train/test split (80/20)X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)print(f"Training set: {X_train.shape[0]} samples")print(f"Test set:     {X_test.shape[0]} samples")

## 4. Model Training & Evaluation

We train three models, each with hyperparameter tuning via GridSearchCV (5-fold CV), then evaluate on the held-out test set using accuracy, precision, recall, F1, confusion matrix, and ROC-AUC.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):    """Evaluate a trained classifier and display metrics, confusion matrix, and ROC curve."""    y_pred = model.predict(X_test)    y_prob = model.predict_proba(X_test)        # Metrics    acc = accuracy_score(y_test, y_pred)    print(f"{'='*50}")    print(f"{model_name} — Accuracy: {acc:.3f}")    print(f"{'='*50}")    print(classification_report(y_test, y_pred, digits=3))        # Confusion matrix + ROC curve side by side    fig, axes = plt.subplots(1, 2, figsize=(13, 5))        # Confusion matrix    cm = confusion_matrix(y_test, y_pred)    sns.heatmap(cm, annot=True, cmap='Reds', fmt='g',                xticklabels=['High', 'Low'], yticklabels=['High', 'Low'], ax=axes[0])    axes[0].set_xlabel('Predicted')    axes[0].set_ylabel('Actual')    axes[0].set_title(f'Confusion Matrix — {model_name}', fontweight='bold')        # ROC curve    label_map = {'High': 0, 'Low': 1}    y_test_bin = np.array([label_map[l] for l in y_test])    fp, tp, _ = roc_curve(y_test_bin, y_prob[:, 1])    model_auc = auc(fp, tp)        axes[1].plot(fp, tp, color='darkorange', lw=2, label=f'ROC curve (AUC = {model_auc:.2f})')    axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--', label='Baseline')    axes[1].set_xlim([0, 1])    axes[1].set_ylim([0, 1])    axes[1].set_xlabel('False Positive Rate')    axes[1].set_ylabel('True Positive Rate')    axes[1].set_title(f'ROC Curve — {model_name}', fontweight='bold')    axes[1].legend(loc='lower right')        plt.tight_layout()    plt.show()        return acc, model_auc

### 4.1 Decision Tree

In [ ]:
# Hyperparameter tuningdt_params = {    'max_depth': [None, 10, 15],    'min_samples_split': [2, 3, 4],    'max_leaf_nodes': [None, 40, 50]}dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=0), dt_params, cv=5)dt_grid.fit(X_train, y_train)print(f"Best parameters: {dt_grid.best_params_}")

In [ ]:
# Train with best parametersdt_model = DecisionTreeClassifier(    max_depth=dt_grid.best_params_['max_depth'],    max_leaf_nodes=dt_grid.best_params_['max_leaf_nodes'],    min_samples_split=dt_grid.best_params_['min_samples_split'],    random_state=0)dt_model.fit(X_train, y_train)dt_acc, dt_auc = evaluate_model(dt_model, X_test, y_test, "Decision Tree")

### 4.2 Random Forest

In [ ]:
# Hyperparameter tuningrf_params = {    'n_estimators': [20, 50, 100],    'max_depth': [None, 100, 200]}rf_grid = GridSearchCV(RandomForestClassifier(random_state=0), rf_params, cv=5)rf_grid.fit(X_train, y_train)print(f"Best parameters: {rf_grid.best_params_}")

In [ ]:
# Train with best parametersrf_model = RandomForestClassifier(    max_depth=rf_grid.best_params_['max_depth'],    n_estimators=rf_grid.best_params_['n_estimators'],    random_state=0)rf_model.fit(X_train, y_train)rf_acc, rf_auc = evaluate_model(rf_model, X_test, y_test, "Random Forest")

### 4.3 Logistic Regression

In [ ]:
# Hyperparameter tuninglr_params = {'max_iter': [50, 100, 500, 1000]}lr_grid = GridSearchCV(LogisticRegression(random_state=0), lr_params, cv=5)lr_grid.fit(X_train, y_train)print(f"Best parameters: {lr_grid.best_params_}")

In [ ]:
# Train with best parameterslr_model = LogisticRegression(    max_iter=lr_grid.best_params_['max_iter'],    random_state=0)lr_model.fit(X_train, y_train)lr_acc, lr_auc = evaluate_model(lr_model, X_test, y_test, "Logistic Regression")

### 4.4 Model Comparison

In [ ]:
# Summary comparison tablecomparison = pd.DataFrame({    'Model': ['Decision Tree', 'Random Forest', 'Logistic Regression'],    'Accuracy': [dt_acc, rf_acc, lr_acc],    'AUC': [dt_auc, rf_auc, lr_auc]}).sort_values('Accuracy', ascending=False)print("Model Comparison (sorted by accuracy):")print(comparison.to_string(index=False))print(f"\n→ Random Forest achieves the best overall performance.")

## 5. Model Explainability

### 5.1 Feature Importance (Random Forest)

In [ ]:
# Extract and plot feature importancesfeature_names = df.drop(['quality', 'rating'], axis=1).columnsimportances = rf_model.feature_importances_feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})feat_df = feat_df.sort_values('Importance', ascending=True)plt.figure(figsize=(8, 6))plt.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue', edgecolor='white')plt.xlabel('Importance', fontsize=12)plt.ylabel('Feature', fontsize=12)plt.title('Feature Importances — Random Forest', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\nFeature Importance Ranking:")for _, row in feat_df.sort_values('Importance', ascending=False).iterrows():    print(f"  {row['Feature']:25s} {row['Importance']:.4f}")

### 5.2 Individual Conditional Expectation (ICE) & Partial Dependence Plot (PDP)ICE plots show how predictions change for individual instances as a single feature varies.  PDP shows the average effect across all instances — revealing the marginal relationship between a feature and the predicted outcome.

In [ ]:
# ICE/PDP helper functiondef create_scoring_dataset(chosen_feature, unique_values, scoring_data):    """Create augmented datasets by varying one feature across all unique values."""    scoring_tables = []    for val in unique_values:        augmented = scoring_data.copy()        augmented[:, chosen_feature] = val        scoring_tables.append(augmented)    return scoring_tables

In [ ]:
# Select alcohol (feature index 10) — the most important predictorselected_idx = 10selected_name = feature_names[selected_idx]unique_vals = sorted(list(set(X[:, selected_idx])))# Generate augmented datasets and score themscoring_data = create_scoring_dataset(selected_idx, unique_vals, X)scored_probs = [rf_model.predict_proba(table) for table in scoring_data]print(f"Feature: {selected_name} (index {selected_idx})")print(f"Unique values: {len(unique_vals)}")print(f"Augmented datasets: {len(scoring_data)}")

In [ ]:
# ICE Plot: individual prediction curvesfig, ax = plt.subplots(figsize=(12, 7))for instance_idx in range(len(scored_probs[0])):    instance_preds = [scored_probs[v][instance_idx][0][0] for v in range(len(unique_vals))]    ax.plot(unique_vals, instance_preds, '-', color='lightsteelblue', alpha=0.3)ax.set_xlabel(f'{selected_name} (normalized)', fontsize=12)ax.set_ylabel('P(High Quality)', fontsize=12)ax.set_title(f'ICE Plot — {selected_name}', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# PDP: average prediction across all instancesavg_preds = [np.mean([scored_probs[v][i][0][0] for i in range(len(scored_probs[0]))])              for v in range(len(unique_vals))]fig, ax = plt.subplots(figsize=(12, 7))ax.plot(unique_vals, avg_preds, 'x-', color='green', markersize=8, linewidth=2)ax.set_xlabel(f'{selected_name} (normalized)', fontsize=12)ax.set_ylabel('Average P(High Quality)', fontsize=12)ax.set_title(f'Partial Dependence Plot — {selected_name}', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# Combined: ICE + PDP overlayrand_instance = 1453  # Fixed for reproducibilityinstance_preds = [scored_probs[v][rand_instance][0][0] for v in range(len(unique_vals))]fig, ax = plt.subplots(figsize=(12, 7))# ICE curves (all instances, light grey)for idx in range(len(scored_probs[0])):    preds = [scored_probs[v][idx][0][0] for v in range(len(unique_vals))]    ax.plot(unique_vals, preds, '-', color='lightgrey', alpha=0.2)# PDP (green)ax.plot(unique_vals, avg_preds, 'x', color='green', markersize=8, label='PDP (average)')# Selected instance (blue)ax.plot(unique_vals, instance_preds, '.', color='blue', markersize=6, label=f'Instance {rand_instance}')ax.set_xlabel(f'{selected_name} (normalized)', fontsize=12)ax.set_ylabel('P(High Quality)', fontsize=12)ax.set_title(f'ICE + PDP — {selected_name}', fontsize=14, fontweight='bold')ax.legend(fontsize=11)plt.tight_layout()plt.show()

## 6. Summary| Aspect | Finding ||---|---|| Best model | **Random Forest** (82.2% accuracy, 0.91 AUC) || Top feature | **Alcohol** (18.3% importance) — higher alcohol → higher quality || 2nd feature | **Sulphates** (13.7%) — positive effect on quality || 3rd feature | **Volatile Acidity** (11.3%) — negative effect on quality || Tuning method | GridSearchCV with 5-fold cross-validation || Explainability | ICE and PDP plots confirm alcohol's monotonic positive effect on quality prediction |### Limitations & Future Work- The binary "High/Low" threshold (quality > 5) is somewhat arbitrary; multi-class or ordinal regression could be explored- More advanced ensemble methods (XGBoost, LightGBM) may yield higher accuracy- Feature engineering (interaction terms, polynomial features) was not attempted- The dataset is relatively small (1,599 samples) — results may not generalize to white wines or other regions